In [1]:
import os
import sys
import copy
import csv
import random
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.sparse.linalg import eigsh
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from catboost import CatBoostRegressor
from torch_geometric.data import Data

In [2]:
import sys

try:
    REPO_ROOT = Path(__file__).resolve().parent
except NameError:
    REPO_ROOT = Path.cwd()

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from Models.GraphGPS import GraphGPSRegressor

In [3]:
DATA_FILE = "merged_file_new_2.csv"  # путь к твоему CSV
ID_COL = "vk_id"
TARGETS = [
    "result/0/Экстраверсия–интроверсия",
    "result/0/Привязанность–обособленность",
    "result/0/Самоконтроль–импульсивность",
    "result/0/Эмоциональная_устойчивость–эмоциональная_неустойчивость",
    "result/0/Экспрессивность–практичность"
]

In [4]:
RANDOM_SEED = 42
TEST_SIZE = 0.20
VAL_SIZE_FROM_REMAIN = 0.25   # после отделения test это даст 60/20/20
KNN_K = 10
LAPLACIAN_DIM = 4
GRAPH_HIDDEN_DIM = 32
GRAPH_EMBED_DIM = 32
GRAPH_NUM_LAYERS = 2
GRAPH_HEADS = 2
GRAPH_DROPOUT = 0.2
GRAPH_DROPEDGE = 0.5
GRAPH_MAX_EPOCHS = 250
GRAPH_PATIENCE = 30
OUTPUT_DIR = REPO_ROOT / "local_experiment_outputs"

In [5]:
def print_header(text: str) -> None:
    print("\n" + "=" * 90)
    print(text)
    print("=" * 90)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [6]:
def read_table_robust(path: str | Path) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path.resolve()}")

    suffix = path.suffix.lower()

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if suffix == ".parquet":
        return pd.read_parquet(path)

    attempts = [
        {"sep": ",", "encoding": "utf-8"},
        {"sep": ";", "encoding": "utf-8"},
        {"sep": "\t", "encoding": "utf-8"},
        {"sep": ",", "encoding": "utf-8-sig"},
        {"sep": ";", "encoding": "utf-8-sig"},
        {"sep": ",", "encoding": "cp1251"},
        {"sep": ";", "encoding": "cp1251"},
        {"sep": "\t", "encoding": "cp1251"},
    ]

    errors = []
    for params in attempts:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", pd.errors.ParserWarning)
                df = pd.read_csv(
                    path,
                    sep=params["sep"],
                    encoding=params["encoding"],
                    engine="python",
                    on_bad_lines="skip",
                )
            if df.shape[1] > 1:
                print(f"Файл прочитан как CSV: sep={repr(params['sep'])}, encoding={params['encoding']}")
                return df
        except Exception as e:
            errors.append(f"{params}: {repr(e)}")

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", pd.errors.ParserWarning)
            df = pd.read_csv(
                path,
                sep=None,
                encoding="utf-8",
                engine="python",
                on_bad_lines="skip",
                quoting=csv.QUOTE_NONE,
            )
        if df.shape[1] > 1:
            print("Файл прочитан в аварийном режиме (QUOTE_NONE).")
            return df
    except Exception as e:
        errors.append(f"fallback: {repr(e)}")

    raise ValueError("Не удалось прочитать файл.\n" + "\n".join(errors[:20]))


def load_single_file_dataset(data_file: str | Path, id_col: str, targets: list[str]):
    df_all = read_table_robust(data_file)

    if id_col not in df_all.columns:
        raise ValueError(
            f"В файле нет колонки '{id_col}'.\n"
            f"Найденные колонки:\n{list(df_all.columns)}"
        )

    missing_targets = [c for c in targets if c not in df_all.columns]
    if missing_targets:
        raise ValueError(
            f"Не найдены target-колонки: {missing_targets}\n"
            f"Найденные колонки:\n{list(df_all.columns)}"
        )

    df_all = df_all.dropna(how="all").copy()
    df_all = df_all.drop_duplicates(subset=[id_col]).copy()

    df_targets = df_all[[id_col] + targets].copy()
    feature_cols = [c for c in df_all.columns if c not in [id_col] + targets]
    df_features_raw = df_all[[id_col] + feature_cols].copy()

    return df_features_raw, df_targets, feature_cols


def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def preprocess_features_ohe(df_features_raw: pd.DataFrame, id_col: str) -> pd.DataFrame:
    df_work = df_features_raw.copy()
    feature_cols = [c for c in df_work.columns if c != id_col]

    if not feature_cols:
        raise ValueError("После исключения id-колонки не осталось признаков.")

    # Удаляем полностью пустые
    all_nan_cols = [c for c in feature_cols if df_work[c].isna().all()]
    if all_nan_cols:
        print(f"Удаляю полностью пустые признаки: {len(all_nan_cols)}")
        df_work = df_work.drop(columns=all_nan_cols)
        feature_cols = [c for c in feature_cols if c not in all_nan_cols]

    # Удаляем константные
    constant_cols = [c for c in feature_cols if df_work[c].nunique(dropna=False) <= 1]
    if constant_cols:
        print(f"Удаляю константные признаки: {len(constant_cols)}")
        df_work = df_work.drop(columns=constant_cols)
        feature_cols = [c for c in feature_cols if c not in constant_cols]

    numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_work[c])]
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]

    transformers = []

    if numeric_cols:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_cols,
        ))

    if categorical_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
                ("onehot", make_onehot_encoder()),
            ]),
            categorical_cols,
        ))

    if not transformers:
        raise ValueError("Не удалось определить признаки для препроцессинга.")

    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
    X_arr = preprocessor.fit_transform(df_work[feature_cols])

    out_cols = preprocessor.get_feature_names_out().tolist()
    cleaned_cols = [c.split("__", 1)[1] if "__" in c else c for c in out_cols]

    df_features = pd.DataFrame(X_arr, columns=cleaned_cols, index=df_work.index)
    df_features.insert(0, id_col, df_work[id_col].values)

    return df_features


def prepare_catboost_features(df_features_raw: pd.DataFrame, id_col: str) -> pd.DataFrame:
    df = df_features_raw.copy()

    feature_cols = [c for c in df.columns if c != id_col]
    all_nan_cols = [c for c in feature_cols if df[c].isna().all()]
    if all_nan_cols:
        df = df.drop(columns=all_nan_cols)

    constant_cols = [c for c in df.columns if c != id_col and df[c].nunique(dropna=False) <= 1]
    if constant_cols:
        df = df.drop(columns=constant_cols)

    feature_cols = [c for c in df.columns if c != id_col]
    numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]

    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())

    for col in categorical_cols:
        df[col] = df[col].fillna("__missing__").astype(str)

    return df


def build_knn_edges(df_features_numeric: pd.DataFrame, id_col: str, k: int = 10) -> pd.DataFrame:
    X_num = df_features_numeric.drop(columns=[id_col]).copy()

    if X_num.shape[1] == 0:
        raise ValueError("После препроцессинга не осталось числовых признаков для построения kNN-графа.")

    ids = df_features_numeric[id_col].to_numpy()
    n_neighbors = min(k + 1, len(df_features_numeric))

    nn_model = NearestNeighbors(n_neighbors=n_neighbors)
    nn_model.fit(X_num)

    _, indices = nn_model.kneighbors(X_num)

    edges = set()
    for i, neighs in enumerate(indices):
        for j in neighs[1:]:
            a, b = sorted((ids[i], ids[j]))
            if a != b:
                edges.add((a, b))

    edges_df = pd.DataFrame(sorted(edges), columns=["source", "target"])
    return edges_df


def make_splits_by_id(ids: pd.Series, random_seed: int = 42):
    ids = pd.Series(ids).drop_duplicates().to_numpy()

    train_val_ids, test_ids = train_test_split(
        ids,
        test_size=TEST_SIZE,
        random_state=random_seed,
        shuffle=True,
    )

    train_ids, val_ids = train_test_split(
        train_val_ids,
        test_size=VAL_SIZE_FROM_REMAIN,
        random_state=random_seed,
        shuffle=True,
    )

    return {
        "train_ids": set(train_ids),
        "val_ids": set(val_ids),
        "test_ids": set(test_ids),
    }


def evaluate_regression(y_true: pd.DataFrame | np.ndarray, y_pred: np.ndarray, targets: list[str], model_name: str, experiment_name: str) -> pd.DataFrame:
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(y_pred)

    r2 = r2_score(y_true_arr, y_pred_arr, multioutput="raw_values")
    rmse = np.sqrt(mean_squared_error(y_true_arr, y_pred_arr, multioutput="raw_values"))
    mae = mean_absolute_error(y_true_arr, y_pred_arr, multioutput="raw_values")

    metrics_df = pd.DataFrame({
        "Experiment": experiment_name,
        "Model": model_name,
        "Target": targets,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
    })

    avg_row = pd.DataFrame([{
        "Experiment": experiment_name,
        "Model": model_name,
        "Target": "MEAN",
        "R2": float(np.mean(r2)),
        "RMSE": float(np.mean(rmse)),
        "MAE": float(np.mean(mae)),
    }])

    return pd.concat([metrics_df, avg_row], ignore_index=True)

In [7]:
class PsychologyGraph:
    def __init__(self, df_nodes, df_edges, targets):
        self.df_nodes = df_nodes.copy()
        self.df_edges = df_edges.copy()
        self.targets_cols = targets

        self.user_ids = self.df_nodes["vk_id"].values
        exclude_cols = ["vk_id"] + targets
        self.node_feature_cols = [c for c in self.df_nodes.columns if c not in exclude_cols]

        self.node_features = self.df_nodes[self.node_feature_cols].values

        n_quantiles_nodes = min(1000, len(self.df_nodes))
        self.node_scaler = QuantileTransformer(n_quantiles=n_quantiles_nodes, random_state=42)
        self.node_features_norm = self.node_scaler.fit_transform(self.node_features)

        self.targets = self.df_nodes[self.targets_cols].values
        n_quantiles_targets = min(1000, len(self.df_nodes))
        self.target_scaler = QuantileTransformer(n_quantiles=n_quantiles_targets, random_state=42)
        self.targets_norm = self.target_scaler.fit_transform(self.targets)

        self.id_to_idx = {uid: i for i, uid in enumerate(self.user_ids)}

    def build_graph(self):
        G = nx.Graph()
        for uid in self.user_ids:
            G.add_node(uid)

        for _, row in self.df_edges.iterrows():
            u = row["source"]
            v = row["target"]
            if u in G and v in G and u != v:
                G.add_edge(u, v)

        for i, uid in enumerate(self.user_ids):
            G.nodes[uid]["features"] = self.node_features_norm[i]

        return G

    def to_pyg_data(self, G, add_pe=True, pe_dim=0):
        node_list = list(G.nodes())
        node_to_idx = {node: idx for idx, node in enumerate(node_list)}

        x = torch.tensor(
            [G.nodes[node]["features"] for node in node_list],
            dtype=torch.float32
        )

        if add_pe and pe_dim > 0:
            pe = self.get_laplacian_pe_from_graph(G, k=pe_dim)
            x = torch.cat([x, pe], dim=-1)

        edge_pairs = [[node_to_idx[u], node_to_idx[v]] for u, v in G.edges()]
        if len(edge_pairs) == 0:
            edge_index = torch.empty((2, 0), dtype=torch.long)
        else:
            edge_index = torch.tensor(edge_pairs, dtype=torch.long).t().contiguous()

        y = torch.tensor(
            [self.targets_norm[self.id_to_idx[node]] for node in node_list],
            dtype=torch.float32
        )

        data = Data(x=x, edge_index=edge_index, y=y)
        data.node_ids = node_list
        return data

    def get_laplacian_pe_from_graph(self, G, k=4):
        num_nodes = len(G.nodes())

        if num_nodes <= 2:
            return torch.zeros((num_nodes, k), dtype=torch.float32)

        try:
            max_k = min(k + 1, num_nodes - 1)
            if max_k <= 1:
                return torch.zeros((num_nodes, k), dtype=torch.float32)

            L = nx.normalized_laplacian_matrix(G).astype(np.float64)
            eig_vals, eig_vecs = eigsh(L, k=max_k, which="SM")
            idx = eig_vals.argsort()
            eig_vecs = eig_vecs[:, idx]

            pe = eig_vecs[:, 1:min(k + 1, eig_vecs.shape[1])]
            if pe.shape[1] < k:
                pad = np.zeros((num_nodes, k - pe.shape[1]))
                pe = np.hstack([pe, pad])

            pe = (pe - pe.mean(axis=0)) / (pe.std(axis=0) + 1e-8)
            return torch.tensor(pe, dtype=torch.float32)

        except Exception as e:
            print(f"Warning: PE computation failed: {e}")
            return torch.zeros((num_nodes, k), dtype=torch.float32)

    def compute_structural_encodings(self, G, laplacian_dim=4):
        node_order = list(self.user_ids)

        degree_dict = dict(G.degree())
        degree_centrality = nx.degree_centrality(G)
        clustering = nx.clustering(G)
        pagerank = nx.pagerank(G)

        try:
            closeness = nx.closeness_centrality(G)
        except Exception:
            closeness = {n: 0.0 for n in G.nodes()}

        try:
            betweenness = nx.betweenness_centrality(G, normalized=True)
        except Exception:
            betweenness = {n: 0.0 for n in G.nodes()}

        try:
            eigenvector = nx.eigenvector_centrality_numpy(G)
        except Exception:
            eigenvector = {n: 0.0 for n in G.nodes()}

        try:
            triangles = nx.triangles(G)
        except Exception:
            triangles = {n: 0 for n in G.nodes()}

        try:
            core_num = nx.core_number(G)
        except Exception:
            core_num = {n: 0 for n in G.nodes()}

        component_size = {}
        for comp in nx.connected_components(G):
            comp_len = len(comp)
            for node in comp:
                component_size[node] = comp_len

        lap_pe = self.get_laplacian_pe_from_graph(G, k=laplacian_dim).cpu().numpy()

        rows = []
        for i, uid in enumerate(node_order):
            row = {
                "vk_id": uid,
                "degree": float(degree_dict.get(uid, 0)),
                "degree_centrality": float(degree_centrality.get(uid, 0.0)),
                "clustering": float(clustering.get(uid, 0.0)),
                "pagerank": float(pagerank.get(uid, 0.0)),
                "closeness_centrality": float(closeness.get(uid, 0.0)),
                "betweenness_centrality": float(betweenness.get(uid, 0.0)),
                "eigenvector_centrality": float(eigenvector.get(uid, 0.0)),
                "triangles": float(triangles.get(uid, 0)),
                "core_number": float(core_num.get(uid, 0)),
                "component_size": float(component_size.get(uid, 1)),
            }

            for j in range(laplacian_dim):
                row[f"lap_pe_{j+1}"] = float(lap_pe[i, j])

            rows.append(row)

        structural_df = pd.DataFrame(rows)

        structural_cols = [c for c in structural_df.columns if c != "vk_id"]
        structural_df[structural_cols] = (
            structural_df[structural_cols]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )

        return structural_df

    def inverse_transform_targets(self, y_pred_norm):
        if isinstance(y_pred_norm, torch.Tensor):
            y_pred_norm = y_pred_norm.detach().cpu().numpy()
        return self.target_scaler.inverse_transform(y_pred_norm)


In [8]:
def train_gradient_boosting(
        df_features: pd.DataFrame,
        df_targets: pd.DataFrame,
        splits: dict,
        id_col: str,
        targets: list[str],
        experiment_name: str,
):
    merged = df_features.merge(df_targets, on=id_col, how="inner")

    train_df = merged[merged[id_col].isin(splits["train_ids"])].copy()
    test_df = merged[merged[id_col].isin(splits["test_ids"])].copy()

    X_train = train_df.drop(columns=[id_col] + targets)
    y_train = train_df[targets]

    X_test = test_df.drop(columns=[id_col] + targets)
    y_test = test_df[targets]

    model = MultiOutputRegressor(
        GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_SEED,
        )
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = evaluate_regression(y_test, y_pred, targets, "GradientBoosting", experiment_name)
    return model, metrics


def train_random_forest(
        df_features: pd.DataFrame,
        df_targets: pd.DataFrame,
        splits: dict,
        id_col: str,
        targets: list[str],
        experiment_name: str,
):
    merged = df_features.merge(df_targets, on=id_col, how="inner")

    train_df = merged[merged[id_col].isin(splits["train_ids"])].copy()
    test_df = merged[merged[id_col].isin(splits["test_ids"])].copy()

    X_train = train_df.drop(columns=[id_col] + targets)
    y_train = train_df[targets]

    X_test = test_df.drop(columns=[id_col] + targets)
    y_test = test_df[targets]

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = evaluate_regression(y_test, y_pred, targets, "RandomForest", experiment_name)
    return model, metrics


def train_catboost(
        df_features_raw: pd.DataFrame,
        df_targets: pd.DataFrame,
        splits: dict,
        id_col: str,
        targets: list[str],
        experiment_name: str,
):
    merged = df_features_raw.merge(df_targets, on=id_col, how="inner")

    train_df = merged[merged[id_col].isin(splits["train_ids"])].copy()
    val_df = merged[merged[id_col].isin(splits["val_ids"])].copy()
    test_df = merged[merged[id_col].isin(splits["test_ids"])].copy()

    X_train = train_df.drop(columns=[id_col] + targets).copy()
    y_train = train_df[targets].copy()

    X_val = val_df.drop(columns=[id_col] + targets).copy()
    y_val = val_df[targets].copy()

    X_test = test_df.drop(columns=[id_col] + targets).copy()
    y_test = test_df[targets].copy()

    cat_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    cat_features = [X_train.columns.get_loc(c) for c in cat_cols]

    model = CatBoostRegressor(
        loss_function="MultiRMSE",
        eval_metric="MultiRMSE",
        iterations=500,
        learning_rate=0.05,
        depth=6,
        random_seed=RANDOM_SEED,
        verbose=False,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_val, y_val),
        use_best_model=True,
        verbose=False,
    )

    y_pred = model.predict(X_test)
    metrics = evaluate_regression(y_test, y_pred, targets, "CatBoost", experiment_name)
    return model, metrics


In [9]:
def make_masks_for_graph_data(data: Data, node_ids: list, splits: dict):
    train_ids = splits["train_ids"]
    val_ids = splits["val_ids"]
    test_ids = splits["test_ids"]

    data.train_mask = torch.tensor([uid in train_ids for uid in node_ids], dtype=torch.bool)
    data.val_mask = torch.tensor([uid in val_ids for uid in node_ids], dtype=torch.bool)
    data.test_mask = torch.tensor([uid in test_ids for uid in node_ids], dtype=torch.bool)
    return data


def train_epoch_graph(model, data, criterion, optimizer, device):
    model.train()
    optimizer.zero_grad()

    x = data.x.to(device)
    edge_index = data.edge_index.to(device)
    y = data.y.to(device)

    out = model(x, edge_index)
    loss = criterion(out[data.train_mask], y[data.train_mask])

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    return loss.item()


def validate_graph(model, data, criterion, device):
    model.eval()
    x = data.x.to(device)
    edge_index = data.edge_index.to(device)
    y = data.y.to(device)

    with torch.no_grad():
        out = model(x, edge_index)
        val_pred = out[data.val_mask]
        val_true = y[data.val_mask]
        loss = criterion(val_pred, val_true).item()

    return loss


def evaluate_graphgps_original_scale(model, data, device, graph_obj: PsychologyGraph, targets: list[str], experiment_name: str):
    model.eval()

    x = data.x.to(device)
    edge_index = data.edge_index.to(device)

    with torch.no_grad():
        out = model(x, edge_index).cpu().numpy()

    y_pred_test_norm = out[data.test_mask.cpu().numpy()]
    y_true_test_norm = data.y[data.test_mask].cpu().numpy()

    y_pred_test = graph_obj.inverse_transform_targets(y_pred_test_norm)
    y_true_test = graph_obj.inverse_transform_targets(y_true_test_norm)

    metrics = evaluate_regression(y_true_test, y_pred_test, targets, "GraphGPS", experiment_name)
    return metrics


def train_graphgps(
        df_features_graph: pd.DataFrame,
        df_targets: pd.DataFrame,
        edges_df: pd.DataFrame,
        splits: dict,
        id_col: str,
        targets: list[str],
        experiment_name: str,
):
    df_nodes = df_features_graph.merge(df_targets, on=id_col, how="inner")

    graph_obj = PsychologyGraph(
        df_nodes=df_nodes,
        df_edges=edges_df,
        targets=targets,
    )

    G = graph_obj.build_graph()
    data = graph_obj.to_pyg_data(G, add_pe=False, pe_dim=0)
    data = make_masks_for_graph_data(data, data.node_ids, splits)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = GraphGPSRegressor(
        node_dim=data.x.shape[1],
        hidden_dim=GRAPH_HIDDEN_DIM,
        embed_dim=GRAPH_EMBED_DIM,
        num_targets=len(targets),
        num_layers=GRAPH_NUM_LAYERS,
        pe_dim=0,
        heads=GRAPH_HEADS,
        dropout=GRAPH_DROPOUT,
        dropedge_p=GRAPH_DROPEDGE,
    ).to(device)

    criterion = nn.HuberLoss(delta=1.0)
    optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer=optimizer,
        mode="min",
        factor=0.5,
        patience=10,
        min_lr=1e-5,
    )

    best_state = copy.deepcopy(model.state_dict())
    best_val = float("inf")
    bad_epochs = 0

    for epoch in range(1, GRAPH_MAX_EPOCHS + 1):
        train_loss = train_epoch_graph(model, data, criterion, optimizer, device)
        val_loss = validate_graph(model, data, criterion, device)
        scheduler.step(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 20 == 0 or epoch == 1:
            print(f"[{experiment_name}][GraphGPS] epoch={epoch:03d} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

        if bad_epochs >= GRAPH_PATIENCE:
            print(f"[{experiment_name}][GraphGPS] early stopping on epoch {epoch}")
            break

    model.load_state_dict(best_state)
    metrics = evaluate_graphgps_original_scale(model, data, device, graph_obj, targets, experiment_name)
    return model, metrics, G, graph_obj

In [10]:
def run_model_suite(
        experiment_name: str,
        df_features_ohe: pd.DataFrame,
        df_features_catboost: pd.DataFrame,
        df_targets: pd.DataFrame,
        edges_df: pd.DataFrame,
        splits: dict,
        id_col: str,
        targets: list[str],
):
    all_metrics = []

    print_header(f"{experiment_name}: Gradient Boosting")
    _, gb_metrics = train_gradient_boosting(
        df_features=df_features_ohe,
        df_targets=df_targets,
        splits=splits,
        id_col=id_col,
        targets=targets,
        experiment_name=experiment_name,
    )
    print(gb_metrics.to_string(index=False))
    all_metrics.append(gb_metrics)

    print_header(f"{experiment_name}: Random Forest")
    _, rf_metrics = train_random_forest(
        df_features=df_features_ohe,
        df_targets=df_targets,
        splits=splits,
        id_col=id_col,
        targets=targets,
        experiment_name=experiment_name,
    )
    print(rf_metrics.to_string(index=False))
    all_metrics.append(rf_metrics)

    print_header(f"{experiment_name}: CatBoost")
    _, cb_metrics = train_catboost(
        df_features_raw=df_features_catboost,
        df_targets=df_targets,
        splits=splits,
        id_col=id_col,
        targets=targets,
        experiment_name=experiment_name,
    )
    print(cb_metrics.to_string(index=False))
    all_metrics.append(cb_metrics)

    print_header(f"{experiment_name}: GraphGPS")
    _, gps_metrics, G, graph_obj = train_graphgps(
        df_features_graph=df_features_ohe,
        df_targets=df_targets,
        edges_df=edges_df,
        splits=splits,
        id_col=id_col,
        targets=targets,
        experiment_name=experiment_name,
    )
    print(gps_metrics.to_string(index=False))
    all_metrics.append(gps_metrics)

    metrics_df = pd.concat(all_metrics, ignore_index=True)
    return metrics_df, G, graph_obj


def main():
    set_seed(RANDOM_SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print_header("Загрузка данных")
    df_features_raw, df_targets, original_feature_cols = load_single_file_dataset(
        DATA_FILE,
        id_col=ID_COL,
        targets=TARGETS,
    )

    print(f"Файл: {Path(DATA_FILE).name}")
    print(f"Строк после очистки: {len(df_features_raw)}")
    print(f"Исходных признаков: {len(original_feature_cols)}")
    print(f"Таргеты: {TARGETS}")

    print_header("Подготовка признаков")
    df_features_ohe = preprocess_features_ohe(df_features_raw, id_col=ID_COL)
    df_features_cat = prepare_catboost_features(df_features_raw, id_col=ID_COL)

    print(f"Признаков после OHE: {len(df_features_ohe.columns) - 1}")
    print(f"Признаков для CatBoost: {len(df_features_cat.columns) - 1}")

    print_header("Разбиение train / val / test")
    splits = make_splits_by_id(df_targets[ID_COL], random_seed=RANDOM_SEED)
    print(f"train: {len(splits['train_ids'])}, val: {len(splits['val_ids'])}, test: {len(splits['test_ids'])}")

    print_header("Построение графа по исходным признакам")
    edges_df = build_knn_edges(df_features_ohe, id_col=ID_COL, k=KNN_K)
    print(f"Рёбер в kNN-графе: {len(edges_df)}")

    # -----------------------------------------------------
    # ЭКСПЕРИМЕНТ 1: только исходные признаки
    # -----------------------------------------------------
    base_metrics, G_base, graph_obj_base = run_model_suite(
        experiment_name="BASE_FEATURES",
        df_features_ohe=df_features_ohe,
        df_features_catboost=df_features_cat,
        df_targets=df_targets,
        edges_df=edges_df,
        splits=splits,
        id_col=ID_COL,
        targets=TARGETS,
    )

    # -----------------------------------------------------
    # ЭКСПЕРИМЕНТ 2: исходные + структурные графовые признаки
    # -----------------------------------------------------
    print_header("Расчёт новых графовых признаков")
    structural_df = graph_obj_base.compute_structural_encodings(G_base, laplacian_dim=LAPLACIAN_DIM)

    print("Новые признаки:")
    print([c for c in structural_df.columns if c != ID_COL])

    df_features_ohe_aug = df_features_ohe.merge(structural_df, on=ID_COL, how="left")
    df_features_cat_aug = df_features_cat.merge(structural_df, on=ID_COL, how="left")

    print(f"Признаков OHE + structural: {len(df_features_ohe_aug.columns) - 1}")
    print(f"Признаков CatBoost + structural: {len(df_features_cat_aug.columns) - 1}")

    aug_metrics, _, _ = run_model_suite(
        experiment_name="BASE_PLUS_STRUCTURAL",
        df_features_ohe=df_features_ohe_aug,
        df_features_catboost=df_features_cat_aug,
        df_targets=df_targets,
        edges_df=edges_df,   # граф оставляем тем же, чтобы сравнение было честным
        splits=splits,
        id_col=ID_COL,
        targets=TARGETS,
    )

    # -----------------------------------------------------
    # СОХРАНЕНИЕ
    # -----------------------------------------------------
    all_metrics = pd.concat([base_metrics, aug_metrics], ignore_index=True)

    base_metrics_path = OUTPUT_DIR / "metrics_base_features.csv"
    aug_metrics_path = OUTPUT_DIR / "metrics_base_plus_structural.csv"
    all_metrics_path = OUTPUT_DIR / "metrics_all.csv"

    structural_path = OUTPUT_DIR / "structural_features.csv"
    ohe_aug_path = OUTPUT_DIR / "dataset_ohe_plus_structural.csv"
    cat_aug_path = OUTPUT_DIR / "dataset_catboost_plus_structural.csv"
    edges_path = OUTPUT_DIR / "knn_edges.csv"

    base_metrics.to_csv(base_metrics_path, index=False)
    aug_metrics.to_csv(aug_metrics_path, index=False)
    all_metrics.to_csv(all_metrics_path, index=False)

    structural_df.to_csv(structural_path, index=False)
    df_features_ohe_aug.to_csv(ohe_aug_path, index=False)
    df_features_cat_aug.to_csv(cat_aug_path, index=False)
    edges_df.to_csv(edges_path, index=False)

    print_header("Итог")
    print(all_metrics.to_string(index=False))
    print(f"\nРезультаты сохранены в: {OUTPUT_DIR.resolve()}")

In [11]:
if __name__ == "__main__":
    main()


Загрузка данных
Файл прочитан как CSV: sep=',', encoding=utf-8
Файл: merged_file_new_2.csv
Строк после очистки: 124
Исходных признаков: 42
Таргеты: ['result/0/Экстраверсия–интроверсия', 'result/0/Привязанность–обособленность', 'result/0/Самоконтроль–импульсивность', 'result/0/Эмоциональная_устойчивость–эмоциональная_неустойчивость', 'result/0/Экспрессивность–практичность']

Подготовка признаков
Удаляю полностью пустые признаки: 2
Удаляю константные признаки: 1
Признаков после OHE: 684
Признаков для CatBoost: 39

Разбиение train / val / test
train: 74, val: 25, test: 25

Построение графа по исходным признакам
Рёбер в kNN-графе: 764

BASE_FEATURES: Gradient Boosting
   Experiment            Model                                                           Target        R2      RMSE       MAE
BASE_FEATURES GradientBoosting                                result/0/Экстраверсия–интроверсия -0.218672 13.603490 10.562786
BASE_FEATURES GradientBoosting                            result/0/Привяза

/var/folders/jr/6ytflvns1_n0vd_mpbq_x8mm0000gn/T/ipykernel_40156/2605285199.py:94: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


   Experiment    Model                                                           Target        R2      RMSE       MAE
BASE_FEATURES CatBoost                                result/0/Экстраверсия–интроверсия -0.161742 13.281944 11.139593
BASE_FEATURES CatBoost                            result/0/Привязанность–обособленность -0.217629 12.302371 10.130320
BASE_FEATURES CatBoost                             result/0/Самоконтроль–импульсивность -0.162071 12.514836 10.140350
BASE_FEATURES CatBoost result/0/Эмоциональная_устойчивость–эмоциональная_неустойчивость -0.033820 11.370808  9.659432
BASE_FEATURES CatBoost                            result/0/Экспрессивность–практичность -0.170495 12.010409 10.005456
BASE_FEATURES CatBoost                                                             MEAN -0.149151 12.296074 10.215030

BASE_FEATURES: GraphGPS


/var/folders/jr/6ytflvns1_n0vd_mpbq_x8mm0000gn/T/ipykernel_40156/3356529541.py:44: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  x = torch.tensor(


[BASE_FEATURES][GraphGPS] epoch=001 train_loss=0.3470 val_loss=0.2687
[BASE_FEATURES][GraphGPS] epoch=020 train_loss=0.0788 val_loss=0.0471
[BASE_FEATURES][GraphGPS] epoch=040 train_loss=0.0637 val_loss=0.0427
[BASE_FEATURES][GraphGPS] early stopping on epoch 57
   Experiment    Model                                                           Target        R2      RMSE       MAE
BASE_FEATURES GraphGPS                                result/0/Экстраверсия–интроверсия -0.315339 14.132719 11.589252
BASE_FEATURES GraphGPS                            result/0/Привязанность–обособленность  0.005400 11.118741  9.154808
BASE_FEATURES GraphGPS                             result/0/Самоконтроль–импульсивность -0.010995 11.673020  9.401475
BASE_FEATURES GraphGPS result/0/Эмоциональная_устойчивость–эмоциональная_неустойчивость -0.140808 11.944698  9.237755
BASE_FEATURES GraphGPS                            result/0/Экспрессивность–практичность -0.065653 11.459903  9.673580
BASE_FEATURES GraphGPS       

/var/folders/jr/6ytflvns1_n0vd_mpbq_x8mm0000gn/T/ipykernel_40156/2605285199.py:94: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


          Experiment    Model                                                           Target        R2      RMSE       MAE
BASE_PLUS_STRUCTURAL CatBoost                                result/0/Экстраверсия–интроверсия -0.074861 12.775651 10.567376
BASE_PLUS_STRUCTURAL CatBoost                            result/0/Привязанность–обособленность -0.010139 11.205261  9.426262
BASE_PLUS_STRUCTURAL CatBoost                             result/0/Самоконтроль–импульсивность  0.022718 11.476747  9.608285
BASE_PLUS_STRUCTURAL CatBoost result/0/Эмоциональная_устойчивость–эмоциональная_неустойчивость -0.011897 11.249598  8.795570
BASE_PLUS_STRUCTURAL CatBoost                            result/0/Экспрессивность–практичность  0.022932 10.973252  9.206366
BASE_PLUS_STRUCTURAL CatBoost                                                             MEAN -0.010249 11.536102  9.520772

BASE_PLUS_STRUCTURAL: GraphGPS
[BASE_PLUS_STRUCTURAL][GraphGPS] epoch=001 train_loss=0.7491 val_loss=0.1014
[BASE_PLUS_STRUC